# Chebyshev Policies and the Mountain Car Problem: Reinforcement Learning for Low-dimensional Control Tasks
## Chebyshev ARS and PPO @ Quanser Aero 2 Environment

Does good performance carry over to a more complicated problem setting?   
We utilize the Quanser Aero 2 environment, which, resembles a typical real-world low-dimensional control task.   

Version 1.0   
Date: 2025-11-17  
Current version: hannes.unger@fh-salzburg.ac.at, georg.schaefer@fh-salzburg.ac.at      

## Observation Space
(base_env.py)
The observation is a dict with shape (3,): 

| Num | Observation               | Min  | Max |
| --- |---------------------------|------|-----|
| 0   | pitch                     | -np.pi / 2 | np.pi / 2 |
| 1   | target                    | -np.pi / 2 | np.pi / 2|
| 2   | velocity                  | -0.24408247 | 0.24408247 |


## Action Space

The action is a ndarray with shape (1,) representing the torque applied to free end of the pendulum. (min = 0, max = 24)  

## Normalization

For norm_action=True, norm_observation=True all dimensions will be normalized to [-1, 1].

## Rewards

The reward function is defined as:  

r = -np.abs(self.target_tilt - self.pitch)  --> **equals rad deviation**

Or if power penalty weight is > 0:  

Pr step (1 - self.power_penalty_weight) * r + self.power_penalty_weight * power_penalty  

## Starting State

Initial pitch can be defined via initial_tilt parameter.  

## Episode End

Truncation: If np.abs(self.pitch) >= np.pi / 2, adds negative reward of (self.stop_time - self.current_time)/sample_time*r   
Termination: After current_time >= stop_time  

In [ ]:
import os
import matplotlib.pyplot as plt
import multiprocessing as mp
import numpy as np
import torch
from itertools import repeat
from aero_envs.utils.evaluation import evaluate_agent
from stable_baselines3 import PPO
from utils import aero, plot

from sb3_contrib import ARS
from polyagents.polynomial_policies import PolynomialARSPolicy, PolynomialPPOPolicy

from gymnasium.wrappers import FlattenObservation

from pickleshare import PickleShareDB
db = PickleShareDB('./picklesharedb')

tensorboard_log_dir = "./tensorboard_logs/aero"
os.makedirs(tensorboard_log_dir, exist_ok=True)

%load_ext autoreload
%autoreload 2

In [ ]:
def plot_axis_min_mean_max_values(ax, sorted_data, names, ylabel, title, color_map):
    for i, (name, value) in enumerate(sorted_data.items()):
        ax.scatter(i, value[0], color=color_map[name], label=name, marker='v')
        ax.scatter(i, value[1], color=color_map[name], label=name, marker='o')
        ax.scatter(i, value[2], color=color_map[name], label=name, marker='^')
    # Add the value next to the point
    if value[1]:
        ax.text(i, value[0], f"{value[0]:.4f}", fontsize=10, verticalalignment='bottom')
        ax.text(i, value[1], f"{value[1]:.4f}", fontsize=10, verticalalignment='bottom')
        ax.text(i, value[2], f"{value[2]:.4f}", fontsize=10, verticalalignment='bottom')

    ax.set_xticks(range(len(names)), names, fontsize=10, rotation=45)
    ax.set_ylabel(ylabel)
    ax.set_title(title, y=1.02)

## Training 

### MLP PPO

In [ ]:
num_cores = mp.cpu_count()
seeds = [0 + i*123 for i in range(num_cores)]
with mp.Pool(processes=num_cores) as pool:
    results = pool.map(aero.run_ppo_training, seeds)
db['aero_ppo_20260113'] = results

Experiment took 152 minutes

In [ ]:
results = db['aero_ppo_20260113']
log_prefix = 'PPO'
log_dir = tensorboard_log_dir + f'/{log_prefix}'
dirs = [log_dir + '/' + r[0] + '_1' for r in results]
plot.plot_tensorboard_rewards_min_mean_max(dirs, title=f'MLP PPO @ Aero 2\nTraining min, mean and max reward over episodes\n12 policies with distinct starting seeds')

### MLP ARS

In [ ]:
num_cores = mp.cpu_count()
seeds = [0 + i*123 for i in range(num_cores)]
with mp.Pool(processes=num_cores) as pool:
    results = pool.map(aero.run_mlp_ars_training, seeds)
db['aero_ars_20260113'] = results

Experiment took 63 minutes

In [ ]:
results = db['aero_ars_20260113']
log_prefix = 'ARS'
log_dir = tensorboard_log_dir + f'/{log_prefix}'
dirs = [log_dir + '/' + r[0] + '_1' for r in results]
plot.plot_tensorboard_rewards_min_mean_max(dirs, title=f'MLP ARS @ Aero 2\nTraining min, mean and max reward over episodes\n12 policies with distinct starting seeds')

### CH ARS

In [ ]:
degree = 3
num_cores = mp.cpu_count()
seeds = [0 + i*123 for i in range(num_cores)]
with mp.Pool(processes=num_cores) as pool:
    results = pool.map(aero.run_ch_ars_training, zip(seeds, repeat(degree)))
db['aero_ch_ars_20260113'] = results

Experiment took 216 minutes

In [ ]:
results = db['aero_ch_ars_20260113']
log_prefix = 'CH_ARS'
log_dir = tensorboard_log_dir + f'/{log_prefix}'
dirs = [log_dir + '/' + r[0] + '_1' for r in results]
plot.plot_tensorboard_rewards_min_mean_max(dirs, title=f'CH ARS @ Aero 2\nTraining min, mean and max reward over episodes\n12 policies with distinct starting seeds')

### CH PPO

In [ ]:
degree = 3
num_cores = mp.cpu_count()
seeds = [0 + i*123 for i in range(num_cores)]
with mp.Pool(processes=num_cores) as pool:
    results = pool.map(aero.run_ch_ppo_training, zip(seeds, repeat(degree)))
db['aero_ch_ppo_20260113'] = results

Experiment took 73 minutes

In [ ]:
results = db['aero_ch_ppo_20260113']
log_prefix = 'CH_PPO'
log_dir = tensorboard_log_dir + f'/{log_prefix}'
dirs = [log_dir + '/' + r[0] + '_1' for r in results]
plot.plot_tensorboard_rewards_min_mean_max(dirs, title=f'CH PPO @ Aero 2\nTraining min, mean and max reward over episodes\n12 policies with distinct starting seeds')

## Evaluation 

#### MLP PPO

In [ ]:
# Determine best agent with respect to deviation
deviations = [row['deviation'] for row in eval_results]
best_agent_index = deviations.index(min(deviations))
db['aero_mlp_ppo_best_agent_params'] = results[best_agent_index][-1]

In [ ]:
results = db['aero_ppo_20260113']
eval_results = []

for r in results:
    model = PPO(env=aero.get_aero_train_env(episodic=True), policy='MultiInputPolicy', device='cpu')
    model.policy.load_state_dict(r[1])

    eval_results.append(evaluate_agent(
        model,
        plot=False,
        num_episodes=10,
        stop_time=200.0,
        seed=42,
    ))

# Determine best agent with respect to deviation
deviations = [row['deviation'] for row in eval_results]
best_agent_index = deviations.index(min(deviations))
db['aero_mlp_ppo_best_agent_params'] = results[best_agent_index][-1]
db['aero_mlp_ppo_best_agent_name'] = results[best_agent_index][0]

# Evaluate best agent in detail
model = PPO(env=aero.get_aero_train_env(episodic=True), policy='MultiInputPolicy', device='cpu')
model.policy.load_state_dict(db['aero_mlp_ppo_best_agent_params'])

db['aero_mlp_ppo_eval_results'] = evaluate_agent(
    model,
    plot=True,
    num_episodes=10,
    stop_time=200.0,
    seed=42,
)

eval_results = db['aero_mlp_ppo_eval_results']

print(f"Deviation: {np.mean(eval_results['deviation']):.4f} ± {np.std(eval_results['deviation']):.4f} rad")
print(f"Power Consumption: {np.mean(eval_results['power']):.4f} ± {np.std(eval_results['power']):.4f} W")
print(f"Action: {np.mean(eval_results['action']):.4f} ± {np.std(eval_results['action']):.4f} V")

The MLP PPO training shows bang-bang type behaviour, as seen in Georg Schäfer's original paper.

### CH PPO

In [ ]:
results = db['aero_ch_ppo_20260113']
eval_results = []

for r in results:
    model = PPO(PolynomialPPOPolicy, env=FlattenObservation(aero.get_aero_train_env(episodic=True)), policy_kwargs=dict(coeffs=r[1]), device='cpu')

    eval_results.append(evaluate_agent(
        model,
        plot=False,
        num_episodes=10,
        stop_time=200.0,
        seed=42,
        flatten_observation=True
    ))

# Determine best agent with respect to deviation
deviations = [row['deviation'] for row in eval_results]
best_agent_index = deviations.index(min(deviations))
db['aero_ch_ppo_best_agent_params'] = results[best_agent_index][-1]
db['aero_ch_ppo_best_agent_name'] = results[best_agent_index][0]

# Evaluate best agent in detail
model = PPO(PolynomialPPOPolicy, env=FlattenObservation(aero.get_aero_train_env(episodic=True)), policy_kwargs=dict(coeffs=db['aero_ch_ppo_best_agent_params']), device='cpu')

db['aero_ch_ppo_eval_results'] = evaluate_agent(
    model,
    plot=True,
    num_episodes=10,
    stop_time=200.0,
    seed=42,
    flatten_observation=True
)

eval_results = db['aero_ch_ppo_eval_results']

print(f"Deviation: {np.mean(eval_results['deviation']):.4f} ± {np.std(eval_results['deviation']):.4f} rad")
print(f"Power Consumption: {np.mean(eval_results['power']):.4f} ± {np.std(eval_results['power']):.4f} W")
print(f"Action: {np.mean(eval_results['action']):.4f} ± {np.std(eval_results['action']):.4f} V")

The CH variant does not show this kind of behaviour.  

### MLP ARS

In [ ]:
results = db['aero_ars_20260113']
eval_results = []

for r in results:
    model = ARS('MlpPolicy', env=FlattenObservation(aero.get_aero_train_env(episodic=True)), zero_policy=False, device='cpu')
    model.policy.action_net.load_state_dict(r[1])

    eval_results.append(evaluate_agent(
        model,
        plot=False,
        num_episodes=10,
        stop_time=200.0,
        seed=42,
        flatten_observation=True
    ))

# Determine best agent with respect to deviation
deviations = [row['deviation'] for row in eval_results]
best_agent_index = deviations.index(min(deviations))
#db['aero_mlp_ars_best_agent_params'] = results[best_agent_index][-1]
db['aero_mlp_ars_best_agent_name'] = results[best_agent_index][0]

In [ ]:
results = db['aero_ars_20260113']
eval_results = []

for r in results:
    model = ARS('MlpPolicy', env=FlattenObservation(aero.get_aero_train_env(episodic=True)), zero_policy=False, device='cpu')
    model.policy.action_net.load_state_dict(r[1])

    eval_results.append(evaluate_agent(
        model,
        plot=False,
        num_episodes=10,
        stop_time=200.0,
        seed=42,
        flatten_observation=True
    ))

# Determine best agent with respect to deviation
deviations = [row['deviation'] for row in eval_results]
best_agent_index = deviations.index(min(deviations))
db['aero_mlp_ars_best_agent_params'] = results[best_agent_index][-1]
db['aero_mlp_ars_best_agent_name'] = results[best_agent_index][0]

# Evaluate best agent in detail
model = ARS('MlpPolicy', env=FlattenObservation(aero.get_aero_train_env(episodic=True)), zero_policy=False, device='cpu')
model.policy.action_net.load_state_dict(db['aero_mlp_ars_best_agent_params'])

db['aero_mlp_ars_eval_results'] = evaluate_agent(
    model,
    plot=True,
    num_episodes=10,
    stop_time=200.0,
    seed=42,
    flatten_observation=True
)

eval_results = db['aero_mlp_ars_eval_results']

print(f"Deviation: {np.mean(eval_results['deviation']):.4f} ± {np.std(eval_results['deviation']):.4f} rad")
print(f"Power Consumption: {np.mean(eval_results['power']):.4f} ± {np.std(eval_results['power']):.4f} W")
print(f"Action: {np.mean(eval_results['action']):.4f} ± {np.std(eval_results['action']):.4f} V")

In [ ]:
results = db['aero_ars_20260113']
rewards = []

for r in results:
    model = ARS(policy='MlpPolicy', env=FlattenObservation(aero.get_aero_train_env(episodic=True)), zero_policy=False, device='cpu', seed=0)
    model.policy.action_net.load_state_dict(r[1])
    rewards.append([r[0], aero.evaluate_aero_agent(model, render=False, flatten_observation=True)])
db['aero_ars_rewards_20260113'] = rewards

Without further hyperparameter tuning, MLP ARS seems to be incapable of adequately learning the required control.  

### CH ARS

In [ ]:
results = db['aero_ch_ars_20260113']
eval_results = []

for r in results:
    model = ARS(PolynomialARSPolicy, env=FlattenObservation(aero.get_aero_train_env(episodic=True)), policy_kwargs=dict(coeffs=r[1]), zero_policy=False, device='cpu')

    eval_results.append(evaluate_agent(
        model,
        plot=False,
        num_episodes=10,
        stop_time=200.0,
        seed=42,
        flatten_observation=True
    ))

# Determine best agent with respect to deviation
deviations = [row['deviation'] for row in eval_results]
best_agent_index = deviations.index(min(deviations))
db['aero_ch_ars_best_agent_params'] = results[best_agent_index][-1]
db['aero_ch_ars_best_agent_name'] = results[best_agent_index][0]

# Evaluate best agent in detail
model = ARS(PolynomialARSPolicy, env=FlattenObservation(aero.get_aero_train_env(episodic=True)), policy_kwargs=dict(coeffs=db['aero_ch_ars_best_agent_params']), zero_policy=False, device='cpu')

db['aero_ch_ars_eval_results'] = evaluate_agent(
    model,
    plot=True,
    num_episodes=10,
    stop_time=200.0,
    seed=42,
    flatten_observation=True
)

eval_results = db['aero_ch_ars_eval_results']

print(f"Deviation: {np.mean(eval_results['deviation']):.4f} ± {np.std(eval_results['deviation']):.4f} rad")
print(f"Power Consumption: {np.mean(eval_results['power']):.4f} ± {np.std(eval_results['power']):.4f} W")
print(f"Action: {np.mean(eval_results['action']):.4f} ± {np.std(eval_results['action']):.4f} V")

Again, the behavior with the CH variant is much better.  

### Comparing Policies

In [ ]:
eval_results_mlp_ppo = db['aero_mlp_ppo_eval_results']
eval_results_ch_ppo = db['aero_ch_ppo_eval_results']

mlp_ppo_deviation = [np.min(eval_results_mlp_ppo['deviation']), np.mean(eval_results_mlp_ppo['deviation']), np.max(eval_results_mlp_ppo['deviation'])]
ch_ppo_deviation = [np.min(eval_results_ch_ppo['deviation']), np.mean(eval_results_ch_ppo['deviation']), np.max(eval_results_ch_ppo['deviation'])]

mlp_ppo_power = [np.min(eval_results_mlp_ppo['power']), np.mean(eval_results_mlp_ppo['power']), np.max(eval_results_mlp_ppo['power'])]
ch_ppo_power = [np.min(eval_results_ch_ppo['power']), np.mean(eval_results_ch_ppo['power']), np.max(eval_results_ch_ppo['power'])]

mlp_ppo_action = [np.min(eval_results_mlp_ppo['action']), np.mean(eval_results_mlp_ppo['action']), np.max(eval_results_mlp_ppo['action'])]
ch_ppo_action = [np.min(eval_results_ch_ppo['action']), np.mean(eval_results_ch_ppo['action']), np.max(eval_results_ch_ppo['action'])]

In [ ]:
deviations = {'ch-3-ppo': ch_ppo_deviation, 'mlp-ppo': mlp_ppo_deviation}
sorted_deviations = dict(sorted(deviations.items(), key=lambda item: item[1][1], reverse=True))

powers = {'ch-3-ppo': ch_ppo_power, 'mlp-ppo': mlp_ppo_power}
sorted_powers = dict(sorted(powers.items(), key=lambda item: item[1][1], reverse=True))

actions = {'ch-3-ppo': ch_ppo_action, 'mlp-ppo': mlp_ppo_action}
sorted_actions = dict(sorted(actions.items(), key=lambda item: item[1][1], reverse=True))

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
fig.set_figwidth(30)
fig.suptitle('PPO Aero 2 evaluation: Min, mean and max values per step\n10 random target trajectories with distinct seeds, duration 200s each', y=1.1)

# Plt1: Extract names and values
deviations_names = list(sorted_deviations.keys())
deviations_values = list(sorted_deviations.values())
powers_names = list(sorted_deviations.keys())
powers_values = list(sorted_deviations.values())
actions_names = list(sorted_deviations.keys())
actions_values = list(sorted_deviations.values())

# Create a consistent color mapping for names
unique_names = list(sorted_deviations.keys())  # Start with names from data1
colors = [plt.cm.tab10(i % 10) for i in range(len(unique_names))]
color_map = {name: colors[i] for i, name in enumerate(unique_names)}

plot_axis_min_mean_max_values(ax1, sorted_deviations, deviations_names, 'rad', 'Pitch deviation', color_map)
plot_axis_min_mean_max_values(ax2, sorted_powers, powers_names, 'W', 'Power consumption', color_map)
plot_axis_min_mean_max_values(ax3, sorted_actions, actions_names, 'V', 'Action magnitude', color_map)

In [ ]:
eval_results_mlp_ars = db['aero_mlp_ars_eval_results']
eval_results_ch_ars = db['aero_ch_ars_eval_results']

mlp_ars_deviation = [np.min(eval_results_mlp_ars['deviation']), np.mean(eval_results_mlp_ars['deviation']), np.max(eval_results_mlp_ars['deviation'])]
ch_ars_deviation = [np.min(eval_results_ch_ars['deviation']), np.mean(eval_results_ch_ars['deviation']), np.max(eval_results_ch_ars['deviation'])]

mlp_ars_power = [np.min(eval_results_mlp_ars['power']), np.mean(eval_results_mlp_ars['power']), np.max(eval_results_mlp_ars['power'])]
ch_ars_power = [np.min(eval_results_ch_ars['power']), np.mean(eval_results_ch_ars['power']), np.max(eval_results_ch_ars['power'])]

mlp_ars_action = [np.min(eval_results_mlp_ars['action']), np.mean(eval_results_mlp_ars['action']), np.max(eval_results_mlp_ars['action'])]
ch_ars_action = [np.min(eval_results_ch_ars['action']), np.mean(eval_results_ch_ars['action']), np.max(eval_results_ch_ars['action'])]

In [ ]:
deviations = {'ch-3-ars': ch_ars_deviation, 'mlp-ars': mlp_ars_deviation}
sorted_deviations = dict(sorted(deviations.items(), key=lambda item: item[1][1], reverse=True))

powers = {'ch-3-ars': ch_ars_power, 'mlp-ars': mlp_ars_power}
sorted_powers = dict(sorted(powers.items(), key=lambda item: item[1][1], reverse=True))

actions = {'ch-3-ars': ch_ars_action, 'mlp-ars': mlp_ars_action}
sorted_actions = dict(sorted(actions.items(), key=lambda item: item[1][1], reverse=True))

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3)
fig.set_figwidth(30)
fig.suptitle('ARS Aero 2 evaluation: Min, mean and max values per step\n10 random target trajectories with distinct seeds, duration 200s each', y=1.1)

# Plt1: Extract names and values
deviations_names = list(sorted_deviations.keys())
deviations_values = list(sorted_deviations.values())
powers_names = list(sorted_deviations.keys())
powers_values = list(sorted_deviations.values())
actions_names = list(sorted_deviations.keys())
actions_values = list(sorted_deviations.values())

# Create a consistent color mapping for names
unique_names = list(sorted_deviations.keys())  # Start with names from data1
colors = [plt.cm.tab10(i % 10) for i in range(len(unique_names))]
color_map = {name: colors[i] for i, name in enumerate(unique_names)}

plot_axis_min_mean_max_values(ax1, sorted_deviations, deviations_names, 'rad', 'Pitch deviation', color_map)
plot_axis_min_mean_max_values(ax2, sorted_powers, powers_names, 'W', 'Power consumption', color_map)
plot_axis_min_mean_max_values(ax3, sorted_actions, actions_names, 'V', 'Action magnitude', color_map)

In [ ]:
model = ARS(PolynomialARSPolicy, env=FlattenObservation(aero.get_aero_train_env(episodic=True)), policy_kwargs=dict(coeffs=db['aero_ch_ars_best_agent_params']), zero_policy=False)
r_chars, obs_chars, act_chars, pow_chars = aero.evaluate_aero_agent(model, render=False, flatten_observation=True, return_obs=True)

model = ARS('MlpPolicy', env=FlattenObservation(aero.get_aero_train_env(episodic=True)), zero_policy=False)
model.policy.action_net.load_state_dict(db['aero_mlp_ars_best_agent_params'])
r_mlpars, obs_mlpars, act_mlpars, pow_mlpars = aero.evaluate_aero_agent(model, render=False, flatten_observation=True, return_obs=True)

model = PPO(PolynomialPPOPolicy, env=FlattenObservation(aero.get_aero_train_env(episodic=True)), policy_kwargs=dict(coeffs=db['aero_ch_ppo_best_agent_params']), device='cpu', seed=0)
r_chppo, obs_chppo, act_chppo, pow_chppo = aero.evaluate_aero_agent(model, render=False, flatten_observation=True, return_obs=True)

model = PPO(env=aero.get_aero_train_env(episodic=True), policy='MultiInputPolicy', device='cpu', seed=0)
model.policy.load_state_dict(db['aero_mlp_ppo_best_agent_params'])
r_mlpppo, obs_mlpppo, act_mlpppo, pow_mlpppo = aero.evaluate_aero_agent(model, render=False, flatten_observation=False, return_obs=True)

In [ ]:
#plot_tilt_series([[f'CH-8 ARS: {r_deg8:.3f}', obs_deg8], [f'CH-6 ARS: {r_deg6:.3f}', obs_deg6], [f'CH-4 ARS: {r_deg4:.3f}', obs_deg4], [f'CH-2 ARS: {r_deg2:.3f}', obs_deg2], [f'PPO: {r_ppo:.3f}', convert_observation_dict_to_arr(obs_ppo)], [f'MLP ARS: {r_mlp:.3f}', obs_mlp]])
aero.plot_tilt_series([[f'MLP PPO', aero.convert_observation_dict_to_arr(obs_mlpppo), aero.extract_action_sequence(act_mlpppo), pow_mlpppo],
                       [f'CH-3 PPO', obs_chppo, aero.extract_action_sequence(act_chppo), pow_chppo],
                       [f'MLP ARS', obs_mlpars, aero.extract_action_sequence(act_mlpars), pow_mlpars],
                       [f'CH-3 ARS', obs_chars, aero.extract_action_sequence(act_chars), pow_chars]])

## Visualizing Policies

In [ ]:
degs = list(range(2, 13, 1))
num_points_per_dim = 100

#Data
xss = np.linspace(-1, 1, num_points_per_dim)
yss = np.linspace(-1, 1, num_points_per_dim)
zss = np.linspace(-1, 1, num_points_per_dim)

X, Y, Z = np.meshgrid(xss, yss, zss)
X_flat = X.flatten()
Y_flat = Y.flatten()
Z_flat = Z.flatten()
points = np.column_stack((X_flat, Y_flat, Z_flat))

In [ ]:
model_ch_ars = ARS(PolynomialARSPolicy, env=FlattenObservation(aero.get_aero_train_env(episodic=True)), policy_kwargs=dict(coeffs=db['aero_ch_ars_best_agent_params']), zero_policy=False)

model_mlp_ars = ARS('MlpPolicy', env=FlattenObservation(aero.get_aero_train_env(episodic=True)), zero_policy=False)
model_mlp_ars.policy.action_net.load_state_dict(db['aero_mlp_ars_best_agent_params'])

model_ch_ppo = PPO(PolynomialPPOPolicy, env=FlattenObservation(aero.get_aero_train_env(episodic=True)), policy_kwargs=dict(coeffs=db['aero_ch_ppo_best_agent_params']), device='cpu', seed=0)

model_mlp_ppo = PPO(env=aero.get_aero_train_env(episodic=True), policy='MultiInputPolicy', device='cpu', seed=0)
model_mlp_ppo.policy.load_state_dict(db['aero_mlp_ppo_best_agent_params'])

In [ ]:
vals_mlp_ppo_unclipped = np.array([model_mlp_ppo.policy._predict(model_mlp_ppo.policy.obs_to_tensor(aero.convert_arr_obs_to_dict_obs(p))[0], deterministic=True).detach().cpu() for p in points]) 
db['aero_mlp_ppo_unclipped_vals'] = vals_mlp_ppo_unclipped

In [ ]:
vals_mlp_ppo_clipped = [model_mlp_ppo.predict(aero.convert_arr_obs_to_dict_obs(p)) for p in points]
db['aero_mlp_ppo_clipped_vals'] = vals_mlp_ppo_clipped

In [ ]:
vals_ch_ppo_unclipped = np.array([model_ch_ppo.policy.policy.policy_approximator.evaluate_point(torch.tensor(p, dtype=torch.float32)) for p in points])
db['aero_ch_ppo_unclipped_vals'] = vals_ch_ppo_unclipped

In [ ]:
vals_ch_ppo_clipped = [model_ch_ppo.predict(torch.tensor(p, dtype=torch.float32)) for p in points]
db['aero_ch_ppo_clipped_vals'] = vals_ch_ppo_clipped

In [ ]:
vals_ch_ars_unclipped = np.array([model_ch_ars.policy.actor.policy.evaluate_point(torch.tensor(p, dtype=torch.float32)) for p in points])
db['aero_ch_ars_unclipped_vals'] = vals_ch_ars_unclipped

In [ ]:
vals_ch_ars_clipped = [model_ch_ars.predict(torch.tensor(p, dtype=torch.float32)) for p in points]
db['aero_ch_ars_clipped_vals'] = vals_ch_ars_clipped

In [ ]:
aero.plot_aero_policy(X_flat, Y_flat, Z_flat, db['aero_mlp_ppo_unclipped_vals'], title='MLP PPO policy (unclipped)')

In [ ]:
aero.plot_aero_policy(X_flat, Y_flat, Z_flat, db['aero_mlp_ppo_clipped_vals'], title='MLP PPO policy')

In [ ]:
aero.plot_aero_policy(X_flat, Y_flat, Z_flat, db['aero_ch_ppo_unclipped_vals'], title='CH-3 PPO policy (unclipped)')

In [ ]:
aero.plot_aero_policy(X_flat, Y_flat, Z_flat, db['aero_ch_ppo_clipped_vals'], title='CH-3 PPO policy')

In [ ]:
aero.plot_aero_policy(X_flat, Y_flat, Z_flat, db['aero_ch_ars_unclipped_vals'], title='CH-3 ARS policy (unclipped)')

In [ ]:
aero.plot_aero_policy(X_flat, Y_flat, Z_flat, db['aero_ch_ars_clipped_vals'], title='CH-3 ARS policy')

## Comparing Sample Efficiency

In [ ]:
db['aero_mlp_ppo_best_agent_name']

In [ ]:
db['aero_ch_ppo_best_agent_name']

In [ ]:
db['aero_mlp_ars_best_agent_name']

In [ ]:
db['aero_ch_ars_best_agent_name']

In [ ]:
tensorboard_dir = tensorboard_log_dir + "/sampleefficiencycomparison"

In [ ]:
def plot_results(
    tensorboard_dir,
    prefixes,
    ax=None,
    title=None,
):
    tag = 'rollout/ep_rew_mean'

    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 8))

    for cfg in prefixes:
        name_prefix = cfg["prefix"]
        label = cfg.get("label", name_prefix)
        color = cfg.get("color", None)

        try:
            dirs = plot.get_logdirs_with_prefix(tensorboard_dir, name_prefix)
            steps, mean, min_vals, max_vals = plot.aggregate_runs(dirs, tag)

            ax.plot(steps, mean, label=label, color=color)
            if min_vals is not None and max_vals is not None:
                ax.fill_between(steps, min_vals, max_vals, alpha=0.3, color=color)
            ax.set_xlabel('Step', fontsize=20)
            ax.set_ylabel('Reward', fontsize=20)
            ax.grid(True)

        except Exception as e:
            print(f"Skipping {name_prefix}: {e}")

    if title is not None:
        ax.set_title(title)

    ax.legend(loc='lower right', fontsize=20)
    ax.tick_params(axis='both', labelsize=20)
    ax.grid(True)

In [ ]:
prefixes = [
    dict(prefix='CH_PPO', label="CH-PPO", color="#1f77b4"),
    dict(prefix='PPO', label="PPO", color="#ff7f0e"),
]

fig, ax1 = plt.subplots() 
fig.set_figwidth(12)
fig.set_figheight(8)
#fig.suptitle(f'PPO @ {env_name}\nMin, mean and max reward over episodes\n{num_policies} policies with distinct starting seeds trained for {n_timesteps} steps')

plot_results(
    tensorboard_dir=tensorboard_dir,
    prefixes=prefixes,
    ax=ax1,
)

plt.savefig("aero_ppo_training_comparison.pdf", bbox_inches='tight')